# FIGS — CSI / Accuracy Verification

Performance diagram (POD vs success ratio with CSI/bias contours), threshold sweep
of CSI/POD/FAR, and AUC/AUPRC — on the held-out **test** split, per hazard.

In [ ]:
# Run with the `met` conda env kernel.
%load_ext autoreload
%autoreload 2          # pick up edits to figs/ without restarting the kernel
import warnings; warnings.filterwarnings('ignore')
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '..')   # so `import figs` works from notebooks/
from figs import config as C

In [ ]:
from figs.data.dataset import read_dataset, feature_columns
from figs.model.wrapper import GBDTModel
from figs.model.calibrate import Calibrator
from figs.config import LEAD_BANDS, HAZARDS
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path

import json
DATA='../Data/processed/figs_v2.parquet'; MODELS='../Data/models'  # DATA must match the models
feats = json.loads((Path(MODELS)/'feature_cols.json').read_text())
df = read_dataset(DATA)  # NOTE: loads the full matrix; heavy at ~5k features
missing = [c for c in feats if c not in df.columns]
if missing: raise ValueError(f'{len(missing)} model features absent from DATA; point DATA at the models\' training set')
te = df[df.split=='test']; print('test rows', len(te))

In [ ]:
def predict_split(h, sub):
    """Calibrated p(h) averaged across covering bands."""
    P = np.zeros(len(sub)); n = np.zeros(len(sub))
    X = sub[feats].to_numpy('float32')
    for b in LEAD_BANDS:
        mp = Path(MODELS)/f'hazard_{h}_{b.name}.pkl'
        if not mp.exists(): continue
        mask = ((sub.fxx>=b.fmin)&(sub.fxx<=b.fmax)).to_numpy() if 'fxx' in sub else np.ones(len(sub),bool)
        if not mask.any(): continue
        p = GBDTModel.load(mp).predict_pos(X[mask])
        cp = Path(MODELS)/f'calib_{h}_{b.name}.pkl'
        if cp.exists(): p = Calibrator.load(cp).transform(p)
        P[mask] += p; n[mask] += 1
    return np.divide(P, n, out=np.zeros_like(P), where=n>0)

## Contingency metrics across probability thresholds

In [ ]:
def sweep(p, y, thr=np.linspace(0.01,0.6,30)):
    rows=[]
    for t in thr:
        pred = p>=t; hit=(pred&(y==1)).sum(); fa=(pred&(y==0)).sum(); miss=((~pred)&(y==1)).sum()
        pod = hit/max(hit+miss,1); far = fa/max(hit+fa,1); sr = 1-far
        csi = hit/max(hit+miss+fa,1); bias=(hit+fa)/max(hit+miss,1)
        rows.append(dict(thr=t,POD=pod,SR=sr,FAR=far,CSI=csi,bias=bias))
    return pd.DataFrame(rows)

sweeps={}
for h in HAZARDS:
    p = predict_split(h, te); y = te[h].to_numpy(int)
    if y.sum()==0: continue
    sweeps[h]=sweep(p,y)
    print(f'{h}: AUC %.3f  AUPRC %.3f  maxCSI %.3f' % (roc_auc_score(y,p), average_precision_score(y,p), sweeps[h].CSI.max()))

## Performance diagram (POD vs success ratio; CSI shaded, bias dashed)

In [ ]:
fig, ax = plt.subplots(figsize=(7,7))
sr = np.linspace(0.001,1,200); pod = np.linspace(0.001,1,200)
SR,POD = np.meshgrid(sr,pod); CSI = 1/(1/SR + 1/POD - 1)
cf = ax.contourf(SR,POD,CSI,levels=np.arange(0,1.01,0.1),cmap='Blues',alpha=0.6)
plt.colorbar(cf,label='CSI')
for bval in [0.5,1,1.5,2,3]:
    ax.plot(sr, bval*sr, 'k--', lw=0.6); ax.text(0.95, min(bval*0.95,0.98), f'{bval}', fontsize=8)
for h,s in sweeps.items():
    ax.plot(s.SR, s.POD, 'o-', ms=3, label=h)
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_xlabel('Success Ratio (1-FAR)'); ax.set_ylabel('POD')
ax.set_title('FIGS performance diagram (test split)'); ax.legend(); plt.show()

## CSI vs threshold

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for h,s in sweeps.items(): ax.plot(s.thr, s.CSI, label=h)
ax.set_xlabel('probability threshold'); ax.set_ylabel('CSI'); ax.legend(); ax.set_title('CSI vs threshold'); plt.show()